In [ ]:
import os
import pandas as pd
from glob import glob
import numpy as np
from functools import reduce
from datetime import datetime
import re
import sys

ndcl_module_path = '/home/data/NDClab/analyses/thrive-theta-ddm/code/python/'
sys.path.append(ndcl_module_path)

from ndclab_py import *

analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"
checked_path = f"{dataset_path}/sourcedata/checked/"
derivatives_path = f"{dataset_path}/derivatives/"
iqs_p_path_s1_r1 = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thriveiqsparent*s1r1*.csv")

ndar_path = find_newest_file(f"/home/data/NDClab/tools/lab-devOps/scripts/ndar_uploads/thrive-dataset/july-2026-nda/all_sessions/ndar_subject01_combined_incomplete.csv")

ndar_data = pd.read_csv(ndar_path, header=1)
ndar_data = ndar_data.pivot(index = "src_subject_id",
    columns="timepoint_label", values=["interview_age"]
)

ndar_data.columns = ndar_data.columns.droplevel(0)
ndar_data["sub"] = ndar_data.index
ndar_data = ndar_data.reset_index(drop=True)
ndar_data = ndar_data[["sub", "s1", "s2", "s3"]]

ndar_data.index.name = None
ndar_data.columns.name = None

# session = "s1_r1"
session = "s1_r1"

if session == "s1_r1":
    session_csv = "s1r1"
    exclude_id_list = [3000242, 3000254, 3000255] # these subs are excluded from ALL behavior + ALL EEG
    unusable_soc = [3000047, 3000056, 3000180, 3000182] # these subs are excluded from SOC behavior + SOC EEG + SOC surveys
    eeg_unusable_nonsoc = [3000008] # these subs are excluded from NONSOC EEG
    exclude_eeg = [3000080, 3000161, 3000228] # these subs are excluded from ALL EEG
    exclude_initstated = [3000066]
    exclude_subset_dyadb = [3000361]
elif session == "s2_r1":
    session_csv = "s2r1"
elif session == "s3_r1":
    session_csv = "s3r1"

csv_output_path = f"{analysis_path}/derivatives/csv/{session}/"

iqs_p_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thriveiqsparent*{session_csv}*.csv")
iqs_ch_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thriveiqschild*{session_csv}*.csv")
bbs_p_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thrivebbsparent*{session_csv}*.csv")
bbs_ch_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thrivebbschild*{session_csv}*.csv")
bbs_ra_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/ThrivebbsRA*{session_csv}*.csv")

behavivor_summary_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/summary*{session}*.csv")
behav_trial_data_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/full_df*.csv")

ern_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_erp_2*.csv")
ern_laplacian_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_erp_laplacian_2*.csv")
tf_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_power_itps*.csv")
icps_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_icps*.csv")

ddm_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/ddm_fit_{session}*.csv")

In [ ]:
age_data = ndar_data[["sub", session[:2]]]
age_data = age_data.rename({session[:2]: "age_m"}, axis=1)

# 1. Load Data
sex_data = pd.read_csv(iqs_p_path_s1_r1)

# 2. Rename ID safely
sex_data = sex_data.rename(columns={"record_id": "sub"})
sex_data["sub"] = sex_data["sub"] - 80000

# 3. Define columns, this must be hardcoded to s1_r1
sex_cols = [c for c in sex_data.columns if f"sexbirth_s1_r1_e1" in c]

# 4. Vectorized Merge (No loops)
# "Take values from English column. If null, fill with Spanish column."
sex_data["sex"] = sex_data[sex_cols].bfill(axis=1).iloc[:, 0]

# 5. Drop rows where sex is still NaN
sex_data = sex_data.dropna(subset=["sex"]).reset_index(drop=True)

# 6. Ensure Integer Type (Safety Step)
# This handles the float '1.0' issue common with NaNs
sex_data["sex"] = sex_data["sex"].astype(int)

# 7. Final Selection
sex_data = sex_data[["sub", "sex"]]

# 1. Load Data
bbs_ra = pd.read_csv(bbs_ra_path)

# 2. Define Columns dynamically
acid_col = f"bbsratrk_acid_{session}_e1"
dpid_col = f"bbsratrk_dpid_{session}_e1"

# 3. Filter for in-person kids (Subject ID check)
# Safety: Ensure we are comparing numbers to numbers
bbs_ra = bbs_ra[bbs_ra[acid_col] >= 3000000].reset_index(drop=True)

# 4. Rename and Deduplicate
bbs_ra = bbs_ra.rename(columns={acid_col: "sub"})
bbs_ra = bbs_ra.drop_duplicates(subset=['sub'], keep='last').reset_index(drop=True)

# 5. Determine Partner Mode (Vectorized & Type-Safe)
# Convert to string first to safely check prefixes
dpid_str = bbs_ra[dpid_col].astype(str)

conditions = [
    dpid_str.str.startswith("300"), # Condition 1: In-person (starts with 300)
    dpid_str.str.startswith("10")   # Condition 2: Remote (starts with 10)
]
choices = [1, 0] # 1 = In-person, 0 = Remote

# np.select is much faster and cleaner than a list comprehension with nested ifs
bbs_ra["dp_inperson"] = np.select(conditions, choices, default=np.nan)

# 6. Final Selection
dp_mode = bbs_ra[["sub", "dp_inperson"]]

# 1. Load Data
full_behavior = pd.read_csv(behav_trial_data_path)

# 2. Extract Condition from the First Trial
first_soc = full_behavior.loc[full_behavior["trial_num"] == 1, ["sub", "condition_soc"]].copy(deep=True)

# 3. Rename for clarity
first_soc = first_soc.rename(columns={"condition_soc": "first_soc"})

# Ensure every subject has exactly one entry for "trial 1". 
# Duplicates imply data errors (e.g., merged files); Missing implies data loss.
if first_soc["sub"].duplicated().any():
    print("Warning: Duplicate trial_num=1 found for some subjects. Check data merging.")
    
first_soc = first_soc.reset_index(drop=True)

# this code creates behavioral data to analyze strictly flanker behavioral measures (hence it doesn't care about unusable EEG-only subjects)
behavior_df = pd.read_csv(behavivor_summary_path)

# subset nonsocial valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]

behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["6_or_more_err_nonsoc"] == 1]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# criteria-based removals
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# subset only data-related columns
behavior_df_nonsoc[[i for i in behavior_df_nonsoc if ("sub" in i or\
                                                    ("acc" in i and "con" in i) or\
                                                     ("peri" in i or "pea" in i or "pes" in i))]]

# subset nonsocial valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]

behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]
behavior_df_soc = behavior_df_soc[behavior_df_soc["6_or_more_err_soc"] == 1]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# criteria-based removals
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

# subset only data-related columns
behavior_df_soc[[i for i in behavior_df_soc if ("sub" in i or\
                                                ("acc" in i and "con" in i) or\
                                                ("peri" in i or "pea" in i or "pes" in i))]]

merged_valid_behav_data = behavior_df_soc.merge(behavior_df_nonsoc, on="sub", how="outer")

merged_valid_behav_data = replace_outliers_with_nan(merged_valid_behav_data, exclude_cols=["sub"])


id_matrix = pd.Series(list(range(3000000, 3000401)), name = "sub")
    
data_frames = [
        id_matrix,
        age_data,
        sex_data,
        first_soc,
        dp_mode,
        # merged_valid_eeg_data,
        merged_valid_behav_data,
        # ddm_df,
        # redcap_data
    ]
    
merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
print(merged_df.shape)

# remove IDs that were not present
merged_df = merged_df.dropna(how="all", subset = [c for c in merged_df if c != "sub"]).reset_index(drop=True)
print(merged_df.shape)

    
id_matrix = pd.Series(list(range(3000000, 3000401)), name = "sub")

iqs_p_data = pd.read_csv(iqs_p_path)
iqs_p_data = iqs_p_data.rename({"record_id" : "sub"}, axis=1)
iqs_p_data["sub"] = iqs_p_data["sub"] - 80000
iqs_p_data = iqs_p_data[
    [i for i in iqs_p_data.columns if (i == "sub" or "scrd" in i)]
]

iqs_ch_data = pd.read_csv(iqs_ch_path)
iqs_ch_data = iqs_ch_data.rename({"record_id" : "sub"}, axis=1)
iqs_ch_data = iqs_ch_data[
    [i for i in iqs_ch_data.columns if (i == "sub" or "scrd" in i)]
]

bbs_p_data = pd.read_csv(bbs_p_path)
bbs_p_data = bbs_p_data.rename({"record_id" : "sub"}, axis=1)
bbs_p_data["sub"] = bbs_p_data["sub"] - 80000
bbs_p_data = bbs_p_data[
    [i for i in bbs_p_data.columns if (i == "sub" or "scrd" in i)]
]

bbs_ch_data = pd.read_csv(bbs_ch_path)
bbs_ch_data = bbs_ch_data.rename({"record_id" : "sub"}, axis=1)
bbs_ch_data = bbs_ch_data[
    [i for i in bbs_ch_data.columns if (i == "sub" or "scrd" in i)]
]

data_frames = [
    id_matrix,
    iqs_p_data,
    iqs_ch_data,
    bbs_p_data,
    bbs_ch_data
]

# the merge is left relative to behav_id
redcap_data = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
# remove IDs that were not present
redcap_data = redcap_data.dropna(how="all", subset = [c for c in redcap_data if c != "sub"]).reset_index(drop=True)
redcap_data.shape[0]

# merge spanish columns with corresponding english columns

# 1. Identify Spanish columns
spanish_columns = [col for col in redcap_data.columns if "es_" in col]

for sp_col in spanish_columns:
    # 2. Derive English column name
    # Your logic: replaces "es_" with "_"
    # Example: "demoes_gender" -> "demo_gender"
    eng_col = "_".join(sp_col.split("es_"))
    
    # Safety Check: Ensure the target English column actually exists
    if eng_col not in redcap_data.columns:
        print(f"[Warning] Could not find English match '{eng_col}' for '{sp_col}'. Skipping.")
        continue

    # 3. Vectorized Merge (No loops)
    redcap_data[eng_col] = redcap_data[eng_col].combine_first(redcap_data[sp_col])

# 4. Cleanup
# Only drop the columns we actually processed
redcap_data = redcap_data.drop(columns=spanish_columns, errors='ignore')

# transform state survey data to reflect the order of conditions (alone/observed) within a session
state_surveys = pd.read_csv(bbs_ch_path)
state_surveys = state_surveys.rename({"record_id": "sub"}, axis=1)

state_surveys = state_surveys[
[i for i in state_surveys.columns if (i == "sub" or "selfnowa" in i or "initstatec" in i or "posttaske" in i or "dyada" in i or "initstated" in i or "posttaskf" in i or "dyadb" in i)\
 and ("timestamp" not in i) and ("_complete" not in i)]
]

state_surveys = state_surveys.merge(first_soc, on="sub", how="left")

new_state_survey_df = pd.DataFrame()
# state_surveys[[i for i in state_surveys.columns if ("initstatec" in i or "first_soc" in i)]]
for c, num_items in zip(["initstatec", "posttaske"], [5, 10]):
    for i in range(state_surveys.shape[0]):
        new_state_survey_df.loc[i, "sub"] = state_surveys.loc[i, "sub"]
        for item in range(1, num_items + 1):
            if state_surveys.loc[i, "first_soc"] == 1:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e1"]
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e2"]
            elif state_surveys.loc[i, "first_soc"] == 0:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e2"]
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e1"]
            else:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = np.nan
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = np.nan

new_state_survey_df = new_state_survey_df.dropna(how="all", subset = new_state_survey_df.columns[1:]).reset_index(drop=True)
state_surveys = state_surveys.dropna(how="all", subset = state_surveys.columns[1:]).reset_index(drop=True)
state_surveys = new_state_survey_df.merge(state_surveys[[i for i in state_surveys.columns if not ("initstatec" in i or "posttaske" in i or "first_soc" in i)]], on="sub", how="left")
print(state_surveys.shape)
redcap_data = redcap_data.merge(state_surveys, on="sub", how="outer")

# --- PART 1: Broad Exclusion (unusable_soc) ---
# Columns to wipe for the "Social" exclusion group
exclude_patterns = ["initstatec", "posttaske", "dyada", "dyadb", "initstated"]
cols_broad = [c for c in redcap_data.columns 
              if np.any([pat in c for pat in exclude_patterns]) 
              and "_nonsoc" not in c]

# Apply NaN for 'unusable_soc' subjects
rows_broad = redcap_data["sub"].isin(unusable_soc)
if cols_broad:
    redcap_data.loc[rows_broad, cols_broad] = np.nan
    print(f"Broad clean: Wiped {len(cols_broad)} columns for subjects {unusable_soc}.")

# --- PART 2: Specific Exclusion (exclude_initstated) ---
# Identify ONLY columns containing 'initstated' (and not nonsocial, if applicable)
cols_init = [c for c in redcap_data.columns 
             if "initstated" in c 
             and "_nonsoc" not in c]

# Apply NaN for 'exclude_initstated' subjects
rows_init = redcap_data["sub"].isin(exclude_initstated)

if cols_init:
    redcap_data.loc[rows_init, cols_init] = np.nan
    print(f"Specific clean: Wiped 'initstated' columns for subjects {exclude_initstated}.")
else:
    print("Warning: No 'initstated' columns found.")


cols_init = [c for c in redcap_data.columns 
             if "initstated" in c 
             and "_nonsoc" not in c]

# Apply NaN for 'exclude_subset_dyadb' subjects
rows_dyadb = redcap_data["sub"].isin(exclude_subset_dyadb)
items_to_exclude = [2, 3, 4, 6, 7, 8]

cols_dyadb = [f"dyadb_i{item}_{session}_e1" for item in items_to_exclude]

if cols_dyadb:
    redcap_data.loc[rows_dyadb, cols_dyadb] = np.nan
    print(f"Specific clean: Wiped 'dyadb' columns for subjects {exclude_subset_dyadb}.")
else:
    print("Warning: No 'dyadb' columns found.")

redcap_data = replace_outliers_with_nan(redcap_data, exclude_cols=["sub"])
print(redcap_data.shape)

In [ ]:
pd.set_option('display.max_columns', None)
state_surveys[~state_surveys['sub'].isin(new_state_survey_df['sub'])].dropna(how='all', subset = state_surveys.columns[1:])

# state_surveys[state_surveys['sub'] == 3000024]

In [ ]:
# sorted(glob.glob('/home/data/NDClab/datasets/thrive-dataset/sourcedata/checked/*'))

In [ ]:
import sys
import os
import glob
import pandas as pd
import numpy as np
import re
import time
import datetime

pd.options.mode.chained_assignment = None

def convert_to_list_rt(series):
    float_list = []
    for value in series:
        if isinstance(value, str):
            if "," in value.strip("[]"):
                float_list.append([float(v) for v in value.strip("[]").split(",")][0]) # Check if the value is a string
            else:
                float_list.append(float(value.strip("[]"))) # Convert string to float and remove brackets
        elif isinstance(value, list): # Check if the value is a list
            float_list.extend([float(v) for v in value]) # Convert each element of the list to float
        else: # Handle NaN values
            float_list.append(np.nan) # Append NaN if value is NaN
    return float_list

def convert_to_list_resp(series):
    resp_list = []
    for value in series:
        if isinstance(value, str):
            converted_row = list(map(int, re.findall(r'\d+', value)))
            resp_list.append(converted_row)
        else: # Handle NaN values
            resp_list.append(np.nan) # Append NaN if value is NaN
    return resp_list


def sort_csvs_by_date_pd(csv_paths):
    date_format = "%Y-%m-%d_%Hh%M.%S.%f"

    def get_first_date(path):
        # Read only the 'date' column from the first row of data
        df = pd.read_csv(path, usecols=['date'], nrows=1)
        # Convert to datetime object for comparison
        return pd.to_datetime(df['date'].iloc[0], format=date_format)

    return sorted(csv_paths, key=get_first_date)

start = time.time()

#session = "s1_r1"
input_dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"
output_dataset_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
data_path = "sourcedata/checked/"
sub_path = f"{session}/psychopy/"
output_path = f"derivatives/behavior/{session}/"

date_time = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
# sys.stdout = open(f"{output_dataset_path}{output_path}{date_time}_log.txt","wt")

n_blocks = 20
n_trials = 40
valid_rt_thresh = 0.150
valid_trial_count = 360
#sub_folders = [i for i in os.listdir(input_dataset_path + data_path) if i.startswith("sub-")]
sub_folders = glob.glob(f"{input_dataset_path}/{data_path}/sub-*/{sub_path}/*")
subjects = sorted(set([re.findall(r'\d+', item.split("/")[-4])[0] for item in sub_folders]))
print(subjects)

for condition in [0, 1]:
    if condition == 0:
        prefix = "_nonsoc"
    elif condition == 1:
        prefix = "_soc"
    # for colname in summary_columns:
        # processing_log[colname + prefix] = []

list_of_trial_data = []

for sub in subjects:
    # processing_log["sub"].append(sub)    
    subject_folder = (input_dataset_path + data_path + "sub-" + sub + os.sep + sub_path)
    num_files = len(os.listdir(subject_folder))
    if (np.any(["no-data" in i for i in os.listdir(subject_folder)])):
        print("sub-{} has no data, skipping...".format(sub))
        continue
    elif ((num_files != 3) or (np.any(["deviation" in i for i in os.listdir(subject_folder)]))):
        print("sub-{} has a deviation in psychopy data ({} files)...".format(sub, num_files))
        sorted_paths = sort_csvs_by_date_pd(glob.glob(f"{subject_folder}/*.csv"))
        data = pd.concat([pd.read_csv(f) for f in sorted_paths], ignore_index=True)
        start_index = data["task_blockText.started"].first_valid_index()
        data = data.iloc[start_index:, :]#.dropna(subset = "middleStim")
        data = data[data["conditionText"].isin(["Observed", "Alone"])].reset_index(drop=True)
        if (data[data["conditionText"] == "Observed"].shape[0] >= valid_trial_count and data[data["conditionText"] == "Alone"].shape[0] >= valid_trial_count):
            print("sub-{} has enough data for both conditions, processing...".format(sub))
            # processing_log["success"].append(1)
        elif (data[data["conditionText"] == "Observed"].shape[0] >= valid_trial_count or data[data["conditionText"] == "Alone"].shape[0] >= valid_trial_count):
            print("sub-{} has enough data for only 1 condition, processing valid condition...".format(sub))
            # processing_log["success"].append(1)
        else:
            print("sub-{} has not enough data for both conditions, skipping...".format(sub))
            # processing_log["success"].append(0)
            continue
    else:
        print("Processing sub-{}...".format(sub))
        # processing_log["success"].append(1)
        
        pattern = f"{subject_folder}/sub-{sub}_arrow-alert-v1-*_psychopy_{session}_e1.csv"
        filename = glob.glob(pattern)
        data = pd.read_csv(filename[0])
        start_index = data["task_blockText.started"].first_valid_index()
        data = data.iloc[start_index:, :]#.dropna(subset = "middleStim")
        data = data[data["conditionText"].isin(["Observed", "Alone"])].reset_index(drop=True)
        # assert (len(data) == n_blocks * n_trials), "Check your data length!"

    # processing_log["n_trials"].append(len(data))
    list_of_trial_data.append(data)

# Create feedback count data

In [ ]:
import pandas as pd

long_data = []
invalid_subs = []

for df in list_of_trial_data:
    subj_col = 'id' if 'id' in df.columns else 'sub'
    subj_id = df[subj_col].iloc[0]
    
    df['blockFeedCat'] = pd.Categorical(df['blockFeedCat'], categories=[1, 2, 3])
    df['conditionText'] = pd.Categorical(df['conditionText'], categories=['Alone', 'Observed']) 
    
    # 1. Count with observed=False to keep empty categories (populates missing with 0)
    counts = df.groupby(['conditionText', 'blockFeedCat'], observed=False).size().reset_index(name='Count')
    
    # EXPLICITLY pass observed=False to silence FutureWarning on Categorical groupby
    counts['Total_Blocks'] = counts.groupby('conditionText', observed=False)['Count'].transform('sum')
    counts['sub'] = subj_id
    
    # --- STRICT VALIDATION CHECKS ---
    condition_totals = counts.groupby('conditionText', observed=False)['Count'].sum()
    total_all_conditions = condition_totals.sum()
    is_valid = True
    
    # Check exactly 10 per condition
    for cond, total in condition_totals.items():
        if total != 10:
            print(f"WARNING: Participant {subj_id} has {total} blocks in condition '{cond}' (Expected: exactly 10).")
            is_valid = False
            
    # Check exactly 20 overall
    if total_all_conditions != 20:
        print(f"WARNING: Participant {subj_id} has {total_all_conditions} total blocks (Expected: exactly 20).")
        is_valid = False
        
    if not is_valid:
        invalid_subs.append(subj_id)
        
    long_data.append(counts)

final_df = pd.concat(long_data, ignore_index=True)
final_df = final_df.rename(columns={'conditionText': 'Condition', 'blockFeedCat': 'Category'})
final_df = final_df[['sub', 'Condition', 'Total_Blocks', 'Category', 'Count']]

# --- FINAL SUMMARY CHECKS ---
print("-" * 50)
if invalid_subs:
    print(f"Subjects failing the strict 10-per-condition / 20-total rule ({len(invalid_subs)} total):")
    print(invalid_subs)
else:
    print("All subjects have exactly 20 blocks (10 per condition).")

# 3. Final Sanity Check
check_df = final_df.groupby(['sub', 'Condition'], observed=False).agg(
    Summed_Count=('Count', 'sum'),
    Listed_Total=('Total_Blocks', 'max') 
).reset_index()

mismatches = check_df[check_df['Summed_Count'] != check_df['Listed_Total']]

print("-" * 50)
if not mismatches.empty:
    print("ERROR: Found mismatches where 'Count' sum != 'Total_Blocks':")
    print(mismatches)
else:
    print("Sanity Check Passed: All 'Count' values add up to 'Total_Blocks' correctly for every participant.")

final_df

In [ ]:
# Extract the valid subject IDs from both dataframes
valid_soc_subs = behavior_df_soc['sub'].unique()
valid_nonsoc_subs = behavior_df_nonsoc['sub'].unique()

# Create boolean masks for both specific rules
mask_observed = (final_df['Condition'] == 'Observed') & (final_df['sub'].isin(valid_soc_subs))
mask_alone = (final_df['Condition'] == 'Alone') & (final_df['sub'].isin(valid_nonsoc_subs))

# Filter final_df by keeping rows that satisfy either mask
filtered_counts_df = final_df[mask_observed | mask_alone]

In [ ]:
mask_alone

In [ ]:
data_frames = [
        filtered_counts_df,
        age_data,
        sex_data,
        dp_mode,
    ]
    
merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
merged_df = merged_df.rename({'Condition': 'soc'}, axis=1)
merged_df['soc'] = [1 if i == 'Observed' else 0 for i in list(merged_df['soc'])]
print(merged_df.shape)
path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/paper_revision/'
merged_df.to_csv(f'{path}/feedback_count_data_{datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")}.csv',
                 index = False)

In [ ]:
merged_df

# Post-task survey (perceived social distress)

In [ ]:
post_task_data = redcap_data[['sub',
# 'posttaske_i1_s1_r1_soc',
 # 'posttaske_i1_s1_r1_nonsoc',
 'posttaske_i2_s1_r1_soc',
 'posttaske_i2_s1_r1_nonsoc',
 'posttaske_i3_s1_r1_soc',
 'posttaske_i3_s1_r1_nonsoc',
 'posttaske_i4_s1_r1_soc',
 'posttaske_i4_s1_r1_nonsoc',
 'posttaske_i5_s1_r1_soc',
 'posttaske_i5_s1_r1_nonsoc',
 'posttaske_i6_s1_r1_soc',
 'posttaske_i6_s1_r1_nonsoc',
 'posttaske_i7_s1_r1_soc',
 'posttaske_i7_s1_r1_nonsoc',
 'posttaske_i8_s1_r1_soc',
 'posttaske_i8_s1_r1_nonsoc',
 'posttaske_i9_s1_r1_soc',
 'posttaske_i9_s1_r1_nonsoc',
 'posttaske_i10_s1_r1_soc',
 'posttaske_i10_s1_r1_nonsoc'
            ]]

post_task_data = post_task_data.dropna(how="all", subset = [c for c in post_task_data if c != "sub"]).reset_index(drop=True)
# redcap_data[~redcap_data['sub'].isin(post_task_data['sub'])]

In [ ]:
import datetime
data_frames = [
        post_task_data,
        age_data,
        sex_data,
        dp_mode,
    ]
    
merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/paper_revision/'
merged_df.to_csv(f'{path}/posttask_raw_data_{datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")}.csv',
                 index = False)

In [ ]:
import pandas as pd

# Create a new dataframe starting with the 'sub' column
agg_df = pd.DataFrame()
agg_df['sub'] = post_task_data['sub']

# Define the exact column names from your post_task_data
f1_soc_cols = ['posttaske_i4_s1_r1_soc', 'posttaske_i5_s1_r1_soc', 'posttaske_i7_s1_r1_soc']
f1_nonsnooc_cols = ['posttaske_i4_s1_r1_nonsoc', 'posttaske_i5_s1_r1_nonsoc', 'posttaske_i7_s1_r1_nonsoc']

f2_soc_cols = ['posttaske_i2_s1_r1_soc', 'posttaske_i6_s1_r1_soc']
f2_nonsoc_cols = ['posttaske_i2_s1_r1_nonsoc', 'posttaske_i6_s1_r1_nonsoc']

# 1. Aggregate items 4, 5, 7 into f1 using SUM (min_count=1 keeps missing data as NaN)
agg_df['f1_soc'] = post_task_data[f1_soc_cols].sum(axis=1, min_count=1)
agg_df['f1_nonsoc'] = post_task_data[f1_nonsoc_cols].sum(axis=1, min_count=1)

# 2. Aggregate items 2, 6 into f2 using SUM (min_count=1 keeps missing data as NaN)
agg_df['f2_soc'] = post_task_data[f2_soc_cols].sum(axis=1, min_count=1)
agg_df['f2_nonsoc'] = post_task_data[f2_nonsoc_cols].sum(axis=1, min_count=1)

# agg_df[agg_df.isnull()]
agg_df

In [ ]:
import pandas as pd

# Use pd.wide_to_long to gather the f1 and f2 columns
long_df = pd.wide_to_long(
    df=agg_df, 
    stubnames=['f1', 'f2'], 
    i='sub', 
    j='condition', 
    sep='_', 
    suffix=r'\w+' # This tells pandas to look for text suffixes like 'soc' and 'nonsoc'
).reset_index()

# Create the binary 'soc' column: 1 if condition is 'soc', else 0
long_df['soc'] = (long_df['condition'] == 'soc').astype(int)

# Drop the text-based 'condition' column since we have the binary 'soc' column now
long_df = long_df.drop(columns=['condition'])

# Optional: Reorder columns for readability
long_df = long_df[['sub', 'soc', 'f1', 'f2']]

long_df

In [ ]:
data_frames = [
        long_df,
        age_data,
        sex_data,
        dp_mode,
    ]
    
merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/paper_revision/'
merged_df.to_csv(f'{path}/posttask_distress_data_{datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")}.csv',
                 index = False)

# Sensitivity analyses for 16 trial cutoff

In [ ]:
# subset subjects who have enough post-error trials
people_passed_cutoff = pd.DataFrame()
upper_cutoff = 99999
lower_cutoff = 16
full_behavior = pd.read_csv(behav_trial_data_path)

counter = 0
for i, sub in enumerate(full_behavior["sub"].unique()):
    # print(sub)
    sub_data = full_behavior[full_behavior["sub"] == sub]
    
    for cond in [0, 1]:
        n_posterr_trials = []
        data = sub_data[sub_data["condition_soc"] == cond]

        n_trials = data[(data["sub"] == sub) & (data["pre_valid_rt"] == 1) & (data["pre_extra_resp"] == 0)\
        & (data["pre_no_resp"] == 0) & (data["pre_congruent"] == 0) & (data["valid_rt"] == 1) & (data["no_resp"] == 0)\
        & (data["pre_accuracy"] == 0)].shape[0]

        if n_trials < upper_cutoff and n_trials >= lower_cutoff:
            people_passed_cutoff.loc[counter, "sub"] = sub
            people_passed_cutoff.loc[counter, "soc"] = cond
            people_passed_cutoff.loc[counter, "n_posterr_trial"] = n_trials
            counter += 1

path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/paper_revision/'
# people_passed_cutoff.to_csv(f'{path}/6_to_15_trials_ids_{datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")}.csv',
#                  index = False)

In [ ]:
# subset subjects who have enough post-error trials
cutoff = 16
full_behavior = pd.read_csv(behav_trial_data_path)

cutoff_results = []

for i, sub in enumerate(full_behavior["sub"].unique()):
    sub_data = full_behavior[full_behavior["sub"] == sub]
    for cond in [0, 1]:
        data = sub_data[sub_data["condition_soc"] == cond]
        n_trials = data[(data["sub"] == sub) & (data["pre_valid_rt"] == 1) & (data["pre_extra_resp"] == 0)\
        & (data["pre_no_resp"] == 0) & (data["pre_congruent"] == 0) & (data["valid_rt"] == 1) & (data["no_resp"] == 0)\
        & (data["pre_accuracy"] == 0)].shape[0]

        cutoff_results.append({
            "sub": sub,
            "soc": cond,
            "n_posterr_trial": n_trials,
            "group": "MANY_ERR" if n_trials >= cutoff else "FEW_ERR"
        })

# Create the raw dataframe
all_people_cutoff = pd.DataFrame(cutoff_results)

# Apply your exact behavioral filtering to this dataframe so it only tracks 
# the subjects who actually made it to this step in your pipeline.
valid_soc_mask = (all_people_cutoff['soc'] == 1) & (all_people_cutoff['sub'].isin(valid_behavior_soc["sub"]))
valid_nonsoc_mask = (all_people_cutoff['soc'] == 0) & (all_people_cutoff['sub'].isin(valid_behavior_nonsoc["sub"]))

# Now all_people_cutoff exactly matches the pool evaluated in your original merge
all_people_cutoff = all_people_cutoff[valid_soc_mask | valid_nonsoc_mask].reset_index(drop=True)

# Split out the passed subjects to continue your pipeline
people_passed_cutoff = all_people_cutoff[all_people_cutoff["group"] == "passed"].reset_index(drop=True)

# Resume your exact merge 
# ddm_df = ddm_df.merge(people_passed_cutoff[['sub', 'soc']], on=['sub', 'soc'], how='inner')

In [ ]:
# reduced_ddm['group'] = 'FEW_ERR'
# full_data['group'] = 'MANY_ERR'
# sensitivity_df = pd.concat([full_data, reduced_ddm]).sort_values(by='sub').reset_index(drop=True)

import datetime

data_frames = [
        all_people_cutoff,
        age_data,
        sex_data,
        # dp_mode,
    ]
    
merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)

path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/paper_revision/'
merged_df.to_csv(f'{path}/sensitivity_data_{datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")}.csv',
                 index = False)

In [ ]:
all_people_cutoff[
(all_people_cutoff["soc"] == 0) & (all_people_cutoff["group"] == "failed")].shape[0]

In [ ]:
all_people_cutoff[
(all_people_cutoff["soc"] == 1) & (all_people_cutoff["group"] == "failed")].shape[0]

In [ ]:
# code to replicate paper values

# here all valid IDs for DDM are created, which is based on behavioral data
behavior_df = pd.read_csv(behavivor_summary_path)

# now, to perform condition-wise outlier removal, we need to subset nonsocial and social conditions and perform removal separately

# subset non-social valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

valid_behavior_nonsoc = behavior_df_nonsoc["sub"].to_frame()

# subset social valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]
behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

valid_behavior_soc = behavior_df_soc["sub"].to_frame()

ddm_df = pd.read_csv(ddm_path)
ddm_df = ddm_df.drop("seed", axis=1)

# subset subjects based on valid behavioral flanker data condition-wise
ddm_df = ddm_df[
    (ddm_df['soc'] == 0) | 
    ((ddm_df['soc'] == 1) & (ddm_df['sub'].isin(valid_behavior_soc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

ddm_df = ddm_df[
    (ddm_df['soc'] == 1) | 
    ((ddm_df['soc'] == 0) & (ddm_df['sub'].isin(valid_behavior_nonsoc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

# subset subjects who have enough post-error trials
people_passed_cutoff = pd.DataFrame()
cutoff = 16

full_behavior = pd.read_csv(behav_trial_data_path)

counter = 0
for i, sub in enumerate(full_behavior["sub"].unique()):
    # print(sub)
    sub_data = full_behavior[full_behavior["sub"] == sub]
    
    for cond in [0, 1]:
        n_posterr_trials = []
        data = sub_data[sub_data["condition_soc"] == cond]

        n_trials = data[(data["sub"] == sub) & (data["pre_valid_rt"] == 1) & (data["pre_extra_resp"] == 0)\
        & (data["pre_no_resp"] == 0) & (data["pre_congruent"] == 0) & (data["valid_rt"] == 1) & (data["no_resp"] == 0)\
        & (data["pre_accuracy"] == 0)].shape[0]

        if n_trials >= cutoff:
            people_passed_cutoff.loc[counter, "sub"] = sub
            people_passed_cutoff.loc[counter, "soc"] = cond
            people_passed_cutoff.loc[counter, "n_posterr_trial"] = n_trials
            counter += 1

# --- ADDED: Track unique subjects before the 16 trial cutoff merge ---
pre_soc_subs = ddm_df[ddm_df['soc'] == 1]['sub'].nunique()
pre_nonsoc_subs = ddm_df[ddm_df['soc'] == 0]['sub'].nunique()

ddm_df = ddm_df.merge(people_passed_cutoff[['sub', 'soc']], on=['sub', 'soc'], how='inner')

# --- ADDED: Calculate and print unique subjects removed after 16 trial cutoff ---
post_soc_subs = ddm_df[ddm_df['soc'] == 1]['sub'].nunique()
post_nonsoc_subs = ddm_df[ddm_df['soc'] == 0]['sub'].nunique()

soc_removed_cutoff = pre_soc_subs - post_soc_subs
nonsoc_removed_cutoff = pre_nonsoc_subs - post_nonsoc_subs
print(f"Unique participants removed by 16 trial cutoff - Social: {soc_removed_cutoff} | Non-social: {nonsoc_removed_cutoff}")

# --- ADDED: Track unique subjects before the fitStat > 200 rejection ---
pre_fit_soc_subs = ddm_df[ddm_df['soc'] == 1]['sub'].nunique()
pre_fit_nonsoc_subs = ddm_df[ddm_df['soc'] == 0]['sub'].nunique()

# remove subjects with bad fit
ddm_df = ddm_df[ddm_df["fitStat"] <= 200].reset_index(drop=True)

# --- ADDED: Calculate and print unique subjects removed after fitStat > 200 ---
post_fit_soc_subs = ddm_df[ddm_df['soc'] == 1]['sub'].nunique()
post_fit_nonsoc_subs = ddm_df[ddm_df['soc'] == 0]['sub'].nunique()

soc_removed_fit = pre_fit_soc_subs - post_fit_soc_subs
nonsoc_removed_fit = pre_fit_nonsoc_subs - post_fit_nonsoc_subs
print(f"Unique participants removed by fitStat > 200 - Social: {soc_removed_fit} | Non-social: {nonsoc_removed_fit}")

print(f"Full DF length: {ddm_df.shape[0]}")

In [ ]:
ddm_cols = [
    'rd_post_trial',
    'rd_diff',
    'reversed_ratio_post_trial',
    'reversed_ratio_diff',
    'sda_post_trial',	
    'sda_diff',
    'ter_post_trial',
    'ter_diff',
    'a_post_trial',
    'a_diff',
    'p_post_trial',
    'p_diff',
]

reduced_ddm = pd.read_csv(find_newest_file(f"{analysis_path}/derivatives/csv/{session}/*reduced*cutoff*.csv"))
pd.options.display.max_columns = None
reduced_ddm = reduced_ddm.dropna(how='all', subset = ddm_cols).reset_index(drop=True)

full_data = pd.read_csv(find_newest_file(f"{analysis_path}/derivatives/csv/{session}/*long*.csv"))
# full_data = full_data[full_data['sub'].isin(people_passed_cutoff['sub'])].reset_index(drop=True)
full_data = full_data[full_data.set_index(['sub', 'soc']).index.isin(people_passed_cutoff.set_index(['sub', 'soc']).index)].reset_index(drop=True)
# assert len(set(full_data['sub']).intersection(set(reduced_ddm['sub']))) == 0, 'Error!'
print(full_data.shape)
assert len(full_data.set_index(['sub', 'soc']).index.intersection(reduced_ddm.set_index(['sub', 'soc']).index)) == 0, 'Error!'

In [ ]:
from scipy import stats
import pandas as pd

# 1. Isolate unique subjects to avoid inflating degrees of freedom
full_unique = full_data.drop_duplicates(subset=['sub'])[['sub', 'age_m', 'sex']].dropna()
reduced_unique = reduced_ddm.drop_duplicates(subset=['sub'])[['sub', 'age_m', 'sex']].dropna()

# 2. T-test for age_m (using Welch's t-test to account for unequal variances)
t_stat, p_val_age = stats.ttest_ind(full_unique['age_m'], reduced_unique['age_m'])
print(f"Age T-test: t = {t_stat:.3f}, p = {p_val_age:.4f}")
print(f"Means -> Full: {full_unique['age_m'].mean():.2f}, Reduced: {reduced_unique['age_m'].mean():.2f}")

# 3. Chi-square test for sex
# Combine into a single dataframe to build the contingency table
full_unique['group'] = 'full'
reduced_unique['group'] = 'reduced'
combined_sex = pd.concat([full_unique, reduced_unique])

contingency_table = pd.crosstab(combined_sex['group'], combined_sex['sex'])
chi2, p_val_sex, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nSex Chi-Square: chi2 = {chi2:.3f}, p = {p_val_sex:.4f}")
print("Contingency Table:\n", contingency_table)

In [ ]:
path = '/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/'
new_df = pd.read_csv('/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/thrive_long_s1_r1_18_09_2026_23_56_31.csv')
old_df = pd.read_csv('/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s1_r1/thrive_long_s1_r1_19_08_2026_15_14_15.csv')

print(
    new_df.shape,
    old_df.shape
)

In [ ]:
[i for i in new_df if "post_trial" in i]

In [ ]:
import pandas as pd

# 1. Identify new columns
new_cols = new_df.columns.difference(old_df.columns).tolist()
print(f"New columns in new_df: {new_cols}")

# 2. Ensure no columns were accidentally dropped
missing_cols = old_df.columns.difference(new_df.columns).tolist()
if missing_cols:
    print(f"Warning: new_df is missing these columns from old_df: {missing_cols}")

# 3. Compare data in common columns
common_cols = old_df.columns.intersection(new_df.columns)

try:
    # .compare() requires identical row indices and handles NaNs natively (NaN == NaN)
    diff = old_df[common_cols].compare(new_df[common_cols])
    
    if diff.empty:
        print("\nAll data in common columns matches perfectly.")
    else:
        print(f"\nFound differences in {len(diff)} rows. Showing differences (self=old, other=new):")
        print(diff)

except ValueError as e:
    print(f"\nComparison failed. Error: {e}")
    print("This usually means the DataFrames have different index labels, a different number of rows, or are sorted differently.")

In [ ]:
# old_df[['sub', 'acc', 'soc', 'rt', 'rt_con', 'rt_err_log']]
for c in ['a', 'fitStat', 'iterNum', 'p', 'rd', 'reversed_ratio', 'sda', 'ter']:
    old_df.rename({c: c+'_post_trial'}, axis=1, inplace=True)


diff = old_df[[
    'a_post_trial', 'fitStat_post_trial', 'iterNum_post_trial',
        'p_post_trial', 'rd_post_trial', 'reversed_ratio_post_trial',
        'sda_post_trial', 'ter_post_trial']].compare(
    new_df[[
    'a_post_trial', 'fitStat_post_trial', 'iterNum_post_trial',
        'p_post_trial', 'rd_post_trial', 'reversed_ratio_post_trial',
        'sda_post_trial', 'ter_post_trial']])

In [ ]:
diff

In [ ]:
sorted(glob(f'{path}/*thrive*long*.csv'))

In [ ]:
reduced_ddm.shape[0] + full_data.shape[0]

In [ ]:
# people_passed_cutoff[people_passed_cutoff['sub'] == 3000010]
reduced_ddm[reduced_ddm['sub'] == 3000010][ddm_cols + ['soc']]

In [ ]:
set(full_data['sub']).intersection(set(reduced_ddm['sub']))

In [ ]:
# here all valid IDs for DDM are created, which is based on behavioral data
behavior_df = pd.read_csv(behavivor_summary_path)

# now, to perform condition-wise outlier removal, we need to subset nonsocial and social conditions and perform removal separately

# subset non-social valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

valid_behavior_nonsoc = behavior_df_nonsoc["sub"].to_frame()

# subset social valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]
behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

valid_behavior_soc = behavior_df_soc["sub"].to_frame()

ddm_df = pd.read_csv(ddm_path)
ddm_df = ddm_df.drop("seed", axis=1)

# subset subjects based on valid behavioral flanker data condition-wise
ddm_df = ddm_df[
    (ddm_df['soc'] == 0) | 
    ((ddm_df['soc'] == 1) & (ddm_df['sub'].isin(valid_behavior_soc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

ddm_df = ddm_df[
    (ddm_df['soc'] == 1) | 
    ((ddm_df['soc'] == 0) & (ddm_df['sub'].isin(valid_behavior_nonsoc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

print(ddm_df.shape)
ddm_df = ddm_df.merge(people_passed_cutoff[['sub', 'soc']], on=['sub', 'soc'], how='inner')
print(ddm_df.shape)

# remove subjects with bad fit
ddm_df = ddm_df[ddm_df["fitStat"] <= 200].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

# convert data to wide format

# ddm_df_soc = ddm_df[ddm_df["soc"] == 1]
# ddm_df_nonsoc = ddm_df[ddm_df["soc"] == 0]

def get_condition_suffix(row):
    acc_str = 'postcorr' if row['pre_accuracy'] == 1 else 'posterr'
    soc_str = 'soc' if row['soc'] == 1 else 'nonsoc'
    return f"{acc_str}_{soc_str}"

ddm_df['condition'] = ddm_df.apply(get_condition_suffix, axis=1)

df_pivoted = ddm_df.pivot(index='sub', columns='condition')

df_pivoted.columns = [f"{col[0]}_{col[1]}" for col in df_pivoted.columns]

df_pivoted.reset_index(inplace=True)

cols_to_drop = [c for c in df_pivoted.columns if c.startswith('pre_accuracy') or c.startswith('soc_')]
ddm_df = df_pivoted.drop(columns=cols_to_drop, errors='ignore')

ddm_df = replace_outliers_with_nan(ddm_df, exclude_cols=["sub"])

# create attentional ratio columns
# Identify all 'Left' columns to drive the loop
rd_cols = [c for c in ddm_df.columns if 'rd_' in c]

for rd_col in rd_cols:
    # Construct the corresponding 'Right' column name
    sda_col = rd_col.replace('rd_', 'sda_')
    
    # Check if the Right pair exists
    if sda_col in ddm_df.columns:
        # Create new name: 
        # 1. Replace '_L_' with '_' (ICPS_soc_early_DLPFC_L_err -> ICPS_soc_early_DLPFC_err)
        # 2. Append '_collapsed'
        new_col = rd_col.replace('rd_', 'reversed_ratio_')
        
        # Calculate mean (NaN + Value = NaN)
        ddm_df[new_col] = (ddm_df[sda_col] / ddm_df[rd_col]) * (-1)

# compute difference scores
# Parameters and conditions to iterate over
params = ['a', 'ter', 'p', 'sda', 'rd', 'reversed_ratio']
conditions = ['soc', 'nonsoc']

for param in params:
    for cond in conditions:
        # Construct existing column names
        posterr_col = f'{param}_posterr_{cond}'
        postcorr_col = f'{param}_postcorr_{cond}'
        
        # Define new difference column name
        # e.g., a_posterr_min_postcorr_soc
        diff_col = f'{param}_diff_{cond}'
        
        # Vectorized subtraction
        # (posterr - postcorr)
        ddm_df[diff_col] = ddm_df[posterr_col] - ddm_df[postcorr_col]

ddm_df = replace_outliers_with_nan_cols(ddm_df, columns_to_check = [c for c in ddm_df if "_diff_" in c])

In [ ]:
(902 - 796) / 4

In [ ]:
for f in glob.glob(f"{input_dataset_path}/{data_path}/sub-*/{sub_path}/*deviation*"):
    with open(f, 'r', encoding='utf-8') as file:
        content = file.read()
        print(content)

In [ ]:
for f in list(glob.glob(f"{input_dataset_path}/{data_path}/sub-*/{sub_path}/*deviation*")):
    print(f)